In [6]:
import os
import glob
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

# Config
fits_folder = "fits"
csv_file = "coordinates.csv"
train_folder = "train"
test_folder = "test"
train_csv = "train.csv"
test_csv = "test.csv"
test_size = 0.2
random_state = 42

os.makedirs(train_folder, exist_ok=True)
os.makedirs(test_folder, exist_ok=True)

# Get all FITS files
fits_files = sorted(glob.glob(os.path.join(fits_folder, "*.fits")))
print(f"Total FITS files: {len(fits_files)}")

# Split files
train_files, test_files = train_test_split(
    fits_files, 
    test_size=test_size, 
    random_state=random_state
)

print(f"Train: {len(train_files)} files")
print(f"Test: {len(test_files)} files")

# Copy files
print("\nCopying files...")
for i, f in enumerate(train_files, 1):
    shutil.copy(f, os.path.join(train_folder, os.path.basename(f)))
    if i % 50 == 0:
        print(f"  Train: {i}/{len(train_files)}")
    
for i, f in enumerate(test_files, 1):
    shutil.copy(f, os.path.join(test_folder, os.path.basename(f)))
    if i % 50 == 0:
        print(f"  Test: {i}/{len(test_files)}")

# Read CSV (skip first row which contains data types)
df = pd.read_csv(csv_file, skiprows=[1])
print(f"\nTotal CSV rows: {len(df)}")

# Extract and normalize Data IDs from filenames
train_ids = []
for f in train_files:
    fname = os.path.basename(f).replace('.fits', '')
    # Convert to lowercase and remove underscores
    fname = fname.lower().replace('_', '')
    train_ids.append(fname)

test_ids = []
for f in test_files:
    fname = os.path.basename(f).replace('.fits', '')
    # Convert to lowercase and remove underscores
    fname = fname.lower().replace('_', '')
    test_ids.append(fname)

print(f"\nSample train IDs: {train_ids[:3]}")
print(f"Sample CSV Data IDs: {df['Data ID'].head(3).tolist()}")

# Filter CSV
train_df = df[df['Data ID'].isin(train_ids)].copy()
test_df = df[df['Data ID'].isin(test_ids)].copy()

train_df.to_csv(train_csv, index=False)
test_df.to_csv(test_csv, index=False)

print(f"\nTrain CSV: {len(train_df)} rows saved to {train_csv}")
print(f"Test CSV: {len(test_df)} rows saved to {test_csv}")

if len(train_df) == 0 or len(test_df) == 0:
    print("\nWARNING: Empty CSV detected!")
    print("Check if filenames match Data IDs in CSV")
print("\nDone!")

Total FITS files: 584
Train: 467 files
Test: 117 files

Copying files...
  Train: 50/467
  Train: 100/467
  Train: 150/467
  Train: 200/467
  Train: 250/467
  Train: 300/467
  Train: 350/467
  Train: 400/467
  Train: 450/467
  Test: 50/117
  Test: 100/117

Total CSV rows: 585

Sample train IDs: ['ic5156', 'pgc07064', 'ngc2639']
Sample CSV Data IDs: ['ngc0024', 'ngc0055', 'ngc0099']

Train CSV: 466 rows saved to train.csv
Test CSV: 117 rows saved to test.csv

Done!
